# orchestration example

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama


In [2]:

class State(TypedDict):
    text: str
    category: str
    draft: str
    final: str

In [3]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

# node 1 : classify

In [ ]:
def classify(state: State):
    prompt = f"""
    Classify the following text into exactly one category:

    news
    email
    other

    Return only the category name.

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    category = response.content.strip().lower()

    return {"category": category}

# node 2 summarize news

In [ ]:
def summarize_news(state: State):
    prompt = f"""
    Summarize this news article concisely.

    Focus on:
    - key event
    - important people or companies
    - business impact

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# node 3 summarize emails

In [ ]:
def summarize_email(state: State):
    prompt = f"""
    Summarize this email.

    Identify:
    - sender's main message
    - requested action
    - any deadline

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# node 4 summarize other text

In [5]:
def summarize_general(state: State):
    prompt = f"""
    Summarize the following text clearly and concisely.

    Text:
    {state["text"]}
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

# routing

In [ ]:
def routeing(state: State):
    category = state["category"]

    if "news" in category:
        return "news"

    elif "email" in category:
        return "email"

    else:
        return "other"